## Setup and Database Connection

The following cell imports the required libraries, establishes a connection to the local MongoDB instance, and defines helper functions for loading and saving collections. These helper functions are used throughout the notebook to keep the code concise and consistent.

In [10]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["mongo_db"]


def load_collection(name):
    df = pd.DataFrame(list(db[name].find()))
    if "_id" in df.columns:
        df = df.drop(columns="_id")
    return df


def save_collection(df, collection_name):
    collection = db[collection_name]
    collection.drop()
    collection.insert_many(df.to_dict("records"))
    print(f"Saved {collection_name} to MongoDB.")

## Data Inspection and Understanding

Before performing any analysis, it is essential to understand the structure and content of the datasets stored in MongoDB. This step helps identify:

- Available variables (columns)
- Data types and potential issues (e.g. numeric values stored as strings)
- Missing or inconsistent values
- Overall size and structure of each dataset

The following code loads each collection from the MongoDB database into a pandas DataFrame and prints:

- Column names → to understand available variables
- First rows → to inspect how the data is structured
- DataFrame info → to check data types and missing values
- Shape → to understand dataset size

This step forms the basis for all further data cleaning and analysis.

In [11]:
collections = [
    "raw_GRP",
    "raw_GRP_per_capita",
    "raw_HPI",
    "raw_birthrate",
    "raw_employment_rate",
    "raw_fertility_rate"
]

for name in collections:
    print("=" * 60)
    print(f"COLLECTION: {name}")
    print("=" * 60)

    df = load_collection(name)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    print(df.head())

    print("\nInfo:")
    df.info()

    print("\nShape:", df.shape)
    print("\n\n")

COLLECTION: raw_GRP

Columns:
['NUTS-Code', 'Region', '2000', '2001.0', '2002.0', '2003.0', '2004.0', '2005.0', '2006.0', '2007.0', '2008.0', '2009.0', '2010.0', '2011.0', '2012.0', '2013.0', '2014.0', '2015.0', '2016.0', '2017.0', '2018.0', '2019.0', '2020.0', '2021.0', '2022.0', '2023.0']

First 5 rows:
  NUTS-Code                   Region    2000    2001.0    2002.0    2003.0  \
0        AT                  Austria  212407  219373.0  225088.0  230542.0   
1     AT111         Mittelburgenland     572     611.0     599.0     636.0   
2     AT112           Nordburgenland    2600    2660.0    2785.0    2847.0   
3     AT113            Südburgenland    1728    1761.0    1821.0    1777.0   
4     AT121  Mostviertel-Eisenwurzen    4558    4699.0    4741.0    4826.0   

     2004.0    2005.0    2006.0    2007.0  ...    2014.0    2015.0    2016.0  \
0  240542.0  252355.0  265934.0  282208.0  ...  330113.0  342084.0  355666.0   
1     645.0     666.0     703.0     747.0  ...     853.0     887

## Cleaning the Birth Rate Dataset

The raw birth rate dataset is stored in long format and already contains the essential analytical dimensions: year, region, and value. However, these fields are still encoded in a non-standard way and must be cleaned before analysis.

The following cleaning steps are performed:

1. The raw collection is loaded from MongoDB.
2. The year is extracted from strings such as `DEMIND_ZEIT-2002`.
3. The regional code is simplified from values such as `GRNU-AT111` to `AT111`.
4. The birth rate values are converted from strings with comma decimal separators (for example `7,260`) into numeric values.
5. The cleaned columns are renamed to more readable names.

This produces a structured dataset that can be used for filtering, grouping, visualization, and later merging with other regional datasets.

In [12]:
birthrate = load_collection("raw_birthrate").copy()

birthrate["year"] = birthrate["C-DEMIND_ZEIT-0"].str.extract(r"(\d{4})").astype("Int64")
birthrate["NUTS-Code"] = birthrate["C-DEMIND_NUTS3-0"].str.replace("GRNU-", "", regex=False)
birthrate["birth_rate"] = pd.to_numeric(
    birthrate["F-DEMIND_CBR"].str.replace(",", ".", regex=False),
    errors="coerce"
)

birthrate = birthrate[birthrate["NUTS-Code"].str.len() == 5]

birthrate_clean = birthrate[["NUTS-Code", "year", "birth_rate"]].copy()
birthrate_clean = birthrate_clean.sort_values(["NUTS-Code", "year"]).reset_index(drop=True)

print(birthrate_clean.head())
birthrate_clean.info()
print(birthrate_clean.isna().sum())
print(birthrate_clean.shape)

  NUTS-Code  year  birth_rate
0     AT111  2002       7.260
1     AT111  2003       6.693
2     AT111  2004       7.744
3     AT111  2005       7.265
4     AT111  2006       6.942
<class 'pandas.DataFrame'>
RangeIndex: 805 entries, 0 to 804
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   NUTS-Code   805 non-null    str    
 1   year        805 non-null    Int64  
 2   birth_rate  805 non-null    float64
dtypes: Int64(1), float64(1), str(1)
memory usage: 19.8 KB
NUTS-Code     0
year          0
birth_rate    0
dtype: int64
(805, 3)


## Storing the Cleaned Birth Rate Dataset

After cleaning and validating the birth rate data, the resulting dataset is stored in MongoDB as a new collection named `clean_birthrate`. This preserves the cleaned version separately from the original raw data and makes it available for later analysis.

In [13]:
save_collection(birthrate_clean, "clean_birthrate")

Saved clean_birthrate to MongoDB.


## Cleaning the GRP Dataset

The raw GRP dataset is stored in wide format, meaning that each year is represented as a separate column. While this format is useful for display, it is not ideal for analysis or merging with other datasets.

To prepare the data for analysis, the dataset is transformed into long format. In the cleaned version, each row represents one observation for a specific region and year.

The following cleaning steps are performed:

1. The raw collection is loaded from MongoDB.
2. Year column names are standardized by removing the `.0` suffix.
3. The dataset is transformed from wide format to long format.
4. The year and GRP values are converted to numeric types.
5. Missing values are removed.
6. Aggregated regions (such as Austria as a whole, e.g. `AT`) are removed to ensure consistency with other datasets that use NUTS3 regional codes.
7. The cleaned dataset is sorted for readability and later merging.

This produces a structured regional time series dataset at the NUTS3 level that can be combined with other cleaned datasets such as the birth rate data.

In [16]:
grp = load_collection("raw_GRP").copy()

# standardize year column names
grp.columns = [col.replace(".0", "") for col in grp.columns]

# transform from wide to long format
grp_clean = grp.melt(
    id_vars=["NUTS-Code", "Region"],
    var_name="year",
    value_name="GRP"
)

# convert to numeric
grp_clean["year"] = pd.to_numeric(grp_clean["year"], errors="coerce").astype("Int64")
grp_clean["GRP"] = pd.to_numeric(grp_clean["GRP"], errors="coerce")

# remove missing values
grp_clean = grp_clean.dropna(subset=["year", "GRP"])

# keep only NUTS3 regions
grp_clean = grp_clean[grp_clean["NUTS-Code"].str.len() == 5]

# sort rows
grp_clean = grp_clean.sort_values(["NUTS-Code", "year"]).reset_index(drop=True)

print(grp_clean.head())
grp_clean.info()
print(grp_clean.isna().sum())
print(grp_clean.shape)

  NUTS-Code            Region  year    GRP
0     AT111  Mittelburgenland  2000  572.0
1     AT111  Mittelburgenland  2001  611.0
2     AT111  Mittelburgenland  2002  599.0
3     AT111  Mittelburgenland  2003  636.0
4     AT111  Mittelburgenland  2004  645.0
<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   NUTS-Code  840 non-null    str    
 1   Region     840 non-null    str    
 2   year       840 non-null    Int64  
 3   GRP        840 non-null    float64
dtypes: Int64(1), float64(1), str(2)
memory usage: 27.2 KB
NUTS-Code    0
Region       0
year         0
GRP          0
dtype: int64
(840, 4)


## Storing the Cleaned GRP Dataset

After cleaning and restructuring the GRP data, the resulting dataset is stored in MongoDB as a new collection named `clean_GRP`. This preserves the cleaned version separately from the raw input data and allows it to be reused in later analysis steps.

In [17]:
save_collection(grp_clean, "clean_GRP")

Saved clean_GRP to MongoDB.
